# Router Component - Hop Count Classification

**Goal:** Classify questions from MetaQA dataset into hop counts (1-hop, 2-hop, or 3-hop).

**Component:** Stage A of the multi-hop question decomposition pipeline

**Jira Task:** TQ-1

**Model:** `Qwen/Qwen3-1.7B` (1.7B parameters, Apache 2.0)

**Approach:** Few-shot classification with concise instruction-following

**Output:** Integer hop count (1, 2, or 3)

## 1. Environment + Versions

In [ ]:
import sys
import torch
import transformers
from pathlib import Path
import json
from datetime import datetime
import random
import re

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Config

In [ ]:
CONFIG = {
    "seed": 42,
    "model_id": "Qwen/Qwen3-1.7B",  # Qwen3 1.7B params
    "data_path": "/kaggle/input/metaqa-questions/refined_1hop.txt",
    "data_path_2hop": "/kaggle/input/metaqa-questions/refined_2hop.txt",
    "data_path_3hop": "/kaggle/input/metaqa-questions/refined_3hop.txt",
    "output_dir": Path("/kaggle/working/runs/router"),
    "run_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    # Generation params
    "max_new_tokens": 64,    # Allow buffer for trace + answer
    "temperature": 0.0,      # Fully deterministic (greedy decoding)
    "approach": "few_shot_direct",  # Method: direct few-shot classification
}

# Set seed for reproducibility
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

print(f"Config: {json.dumps(CONFIG, indent=2, default=str)}")
print(f"Seed: {CONFIG['seed']}")
print(f"Approach: {CONFIG['approach']}")

## 3. Data Loading

def load_questions(file_path):
    """Load questions from a text file (one per line)."""
    path = Path(file_path)
    if not path.exists():
        print(f"Warning: {file_path} not found. Using placeholder data.")
        return []
    
    with open(path, 'r', encoding='utf-8') as f:
        questions = [line.strip() for line in f if line.strip()]
    return questions

# Load test questions
# Note: Update paths when MetaQA dataset is uploaded to Kaggle
questions_1hop = load_questions(CONFIG["data_path"])
questions_2hop = load_questions(CONFIG["data_path_2hop"])
questions_3hop = load_questions(CONFIG["data_path_3hop"])

print(f"Loaded {len(questions_1hop)} 1-hop questions")
print(f"Loaded {len(questions_2hop)} 2-hop questions")
print(f"Loaded {len(questions_3hop)} 3-hop questions")

# Example questions for testing if files not found
if not questions_1hop:
    questions_1hop = ["What does Grégoire Colin appear in?"]
if not questions_2hop:
    questions_2hop = ["which person directed the movies starred by John Krasinski"]
if not questions_3hop:
    questions_3hop = ["the films that share directors with the film Catch Me If You Can were in which languages"]

In [ ]:
def load_questions(file_path):
    """Load questions from a text file (one per line)."""
    path = Path(file_path)
    if not path.exists():
        print(f"Warning: {file_path} not found. Using placeholder data.")
        return []
    
    with open(path, 'r', encoding='utf-8') as f:
        questions = [line.strip() for line in f if line.strip()]
    return questions

# Load test questions
# Note: Update paths when MetaQA dataset is uploaded to Kaggle
questions_1hop = load_questions(CONFIG["data_path"])
questions_2hop = load_questions(CONFIG["data_path_2hop"])
questions_3hop = load_questions(CONFIG["data_path_3hop"])

print(f"Loaded {len(questions_1hop)} 1-hop questions")
print(f"Loaded {len(questions_2hop)} 2-hop questions")
print(f"Loaded {len(questions_3hop)} 3-hop questions")

# Example questions for testing if files not found
if not questions_1hop:
    questions_1hop = ["What does Grégoire Colin appear in?"]
if not questions_2hop:
    questions_2hop = ["which person directed the movies starred by John Krasinski"]
if not questions_3hop:
    questions_3hop = ["the films that share directors with the film Catch Me If You Can were in which languages"]

## 4. Model Setup

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading model: {CONFIG['model_id']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"])

# Load model with proper device handling
if CONFIG["device"] == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_id"],
        torch_dtype=torch.float16,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_id"],
        torch_dtype=torch.float32,
    ).to(CONFIG["device"])

model.eval()
print("Model loaded successfully")

## 5. Router Function

In [ ]:
def classify_hop_count(question, model, tokenizer, device):
    """
    Classify a question into hop count (1, 2, or 3).
    
    Args:
        question: Input question string
        model: Loaded model
        tokenizer: Loaded tokenizer
        device: Device to run inference on
    
    Returns:
        int: Hop count (1, 2, or 3)
    """
    # Show reasoning traces so model understands the counting
    prompt = f"""Classify the question complexity. Output: 1, 2, or 3

Q: What genre is Inception?
Trace: Inception → genre
A: 1

Q: Who directed Titanic?
Trace: Titanic → director
A: 1

Q: What movies did Tom Hanks act in?
Trace: Tom Hanks → movies
A: 1

Q: Who directed movies starring Brad Pitt?
Trace: Brad Pitt → movies → directors
A: 2

Q: What genres are films directed by Nolan?
Trace: Nolan → films → genres
A: 2

Q: What release years are movies starring Emma Watson?
Trace: Emma Watson → movies → release years
A: 2

Q: The director of Avatar directed which other movies?
Trace: Avatar → director → other movies
A: 2

Q: What languages are films that share directors with The Matrix?
Trace: The Matrix → director → other films → languages
A: 3

Q: Who acted in movies whose directors also directed Inception?
Trace: Inception → director → other movies → actors
A: 3

Q: {question}
A:"""
        # Ensure pad_token is set for tokenizer
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,  # Allow buffer for trace + answer
            temperature=CONFIG["temperature"],
            do_sample=False,  # Deterministic
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    
    # Extract number from response
    try:
        # Remove thinking blocks if present (Qwen3 may include <think>...</think>)
        clean_response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
        
        # Take first line only
        clean_response = clean_response.split('\n')[0].strip()
        
        # Try to find a digit 1, 2, or 3
        match = re.search(r'[123]', clean_response)
        if match:
            return int(match.group())
        
        # Try spelled-out numbers
        lower_resp = clean_response.lower()
        if 'one' in lower_resp or '1-hop' in lower_resp:
            return 1
        if 'two' in lower_resp or '2-hop' in lower_resp:
            return 2
        if 'three' in lower_resp or '3-hop' in lower_resp:
            return 3
        
        # Last resort: look for any digit and clamp
        numbers = re.findall(r'\d+', clean_response)
        if numbers:
            return max(1, min(3, int(numbers[0])))
        
        # Default to 2 (middle value) if nothing found
        print(f"Warning: No hop count in response: '{response[:80]}'. Defaulting to 2.")
        return 2
    except Exception as e:
        print(f"Error parsing response '{response[:50]}...': {e}. Defaulting to 2.")
        return 2

## 6. Run / Inference

In [ ]:
# Optional: Batch classification with progress
def classify_batch(questions, model, tokenizer, device, show_progress=True):
    """Classify multiple questions with optional progress display."""
    results = []
    for i, q in enumerate(questions):
        if show_progress and (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(questions)}...")
        hop = classify_hop_count(q, model, tokenizer, device)
        results.append(hop)
    return results


In [ ]:
# Test on sample questions
print("=" * 60)
print("Testing router on sample questions")
print("=" * 60)

test_questions = [
    (questions_1hop[0] if questions_1hop else "What does Grégoire Colin appear in?", 1),
    (questions_2hop[0] if questions_2hop else "which person directed the movies starred by John Krasinski", 2),
    (questions_3hop[0] if questions_3hop else "the films that share directors with the film Catch Me If You Can were in which languages", 3),
]

results = []
for question, expected_hop in test_questions:
    print("\n" + "-" * 60)
    print(f"Q: {question}")
    hop_count = classify_hop_count(question, model, tokenizer, CONFIG["device"])
    correct = hop_count == expected_hop
    results.append({
        "question": question,
        "predicted_hop": hop_count,
        "expected_hop": expected_hop,
        "correct": correct,
    })
    print(f"Predicted: {hop_count}, Expected: {expected_hop}, Correct: {correct}")
    if not correct:
        print(">>> MISMATCH <<<")

# Summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
correct_count = sum(1 for r in results if r["correct"])
print(f"Sample accuracy: {correct_count}/{len(results)} ({100*correct_count/len(results):.1f}%)")

## 7. Evaluation + Metrics

In [ ]:
# Run on all loaded questions
all_questions = []
all_expected = []

if questions_1hop:
    all_questions.extend(questions_1hop)
    all_expected.extend([1] * len(questions_1hop))
if questions_2hop:
    all_questions.extend(questions_2hop)
    all_expected.extend([2] * len(questions_2hop))
if questions_3hop:
    all_questions.extend(questions_3hop)
    all_expected.extend([3] * len(questions_3hop))

print(f"Running evaluation on {len(all_questions)} questions...")
print(f"  1-hop: {len(questions_1hop)}, 2-hop: {len(questions_2hop)}, 3-hop: {len(questions_3hop)}")
print()

# Use batch classification
predictions = classify_batch(all_questions, model, tokenizer, CONFIG["device"])

# Calculate metrics
correct = sum(1 for p, e in zip(predictions, all_expected) if p == e)
accuracy = correct / len(predictions) if predictions else 0.0

# Per-hop accuracy
hop_1_correct = sum(1 for p, e in zip(predictions, all_expected) if e == 1 and p == e)
hop_1_total = sum(1 for e in all_expected if e == 1)
hop_1_acc = hop_1_correct / hop_1_total if hop_1_total > 0 else 0.0

hop_2_correct = sum(1 for p, e in zip(predictions, all_expected) if e == 2 and p == e)
hop_2_total = sum(1 for e in all_expected if e == 2)
hop_2_acc = hop_2_correct / hop_2_total if hop_2_total > 0 else 0.0

hop_3_correct = sum(1 for p, e in zip(predictions, all_expected) if e == 3 and p == e)
hop_3_total = sum(1 for e in all_expected if e == 3)
hop_3_acc = hop_3_correct / hop_3_total if hop_3_total > 0 else 0.0

# Confusion analysis
from collections import Counter
confusion = Counter()
for p, e in zip(predictions, all_expected):
    confusion[(e, p)] += 1

metrics = {
    "overall_accuracy": accuracy,
    "total_questions": len(predictions),
    "correct_predictions": correct,
    "hop_1_accuracy": hop_1_acc,
    "hop_1_total": hop_1_total,
    "hop_2_accuracy": hop_2_acc,
    "hop_2_total": hop_2_total,
    "hop_3_accuracy": hop_3_acc,
    "hop_3_total": hop_3_total,
    "seed": CONFIG["seed"],
    "model": CONFIG["model_id"],
    "run_id": CONFIG["run_id"],
    "approach": CONFIG.get("approach", "few_shot_direct"),
}

print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Overall Accuracy: {accuracy:.4f} ({correct}/{len(predictions)})")
print(f"1-hop Accuracy:   {hop_1_acc:.4f} ({hop_1_correct}/{hop_1_total})")
print(f"2-hop Accuracy:   {hop_2_acc:.4f} ({hop_2_correct}/{hop_2_total})")
print(f"3-hop Accuracy:   {hop_3_acc:.4f} ({hop_3_correct}/{hop_3_total})")

print("\n--- Confusion Matrix (Expected → Predicted) ---")
print("     Pred→   1    2    3")
for expected in [1, 2, 3]:
    row = f"True {expected}:"
    for predicted in [1, 2, 3]:
        count = confusion.get((expected, predicted), 0)
        row += f" {count:4d}"
    print(row)

## 8. Save Artifacts

In [ ]:
# Save Artifacts
output_dir = CONFIG["output_dir"] / CONFIG["run_id"]
output_dir.mkdir(parents=True, exist_ok=True)

# Save metrics
with open(output_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save config snapshot
with open(output_dir / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

# Save notes
notes = f"""Router Component Run - {CONFIG['run_id']}

Model: {CONFIG['model_id']}
Seed: {CONFIG['seed']}
Approach: {CONFIG.get('approach', 'few_shot_direct')}

Results:
- Overall Accuracy: {metrics['overall_accuracy']:.4f}
- 1-hop Accuracy: {metrics['hop_1_accuracy']:.4f} ({metrics['hop_1_total']} questions)
- 2-hop Accuracy: {metrics['hop_2_accuracy']:.4f} ({metrics['hop_2_total']} questions)
- 3-hop Accuracy: {metrics['hop_3_accuracy']:.4f} ({metrics['hop_3_total']} questions)

Method Notes:
- Few-shot classification with explicit hop counting definitions
- Prompt teaches HOW to count hops (intermediate lookups, not steps)
- Key clarifications: final answer step does NOT count, property of step 1 = 2-hop
- 13 diverse examples covering edge cases (same, also, properties)
- Qwen2.5-0.5B-Instruct with greedy decoding for determinism

What to check:
- Review per-hop accuracy to identify which hop counts are harder to classify
- Check detailed_results.json for specific misclassification patterns

Next steps:
- If accuracy is acceptable (>85%), proceed to Decomposer component (TQ-2)
- If accuracy is low, consider:
  * Adding more few-shot examples for edge cases
  * Trying Qwen2.5-1.5B-Instruct (larger variant)
  * Fine-tuning on MetaQA examples
"""

with open(output_dir / "notes.md", "w") as f:
    f.write(notes)

# Save detailed results
detailed_results = [
    {
        "question": q,
        "expected_hop": e,
        "predicted_hop": p,
        "correct": p == e
    }
    for q, e, p in zip(all_questions, all_expected, predictions)
]

with open(output_dir / "detailed_results.json", "w") as f:
    json.dump(detailed_results, f, indent=2, ensure_ascii=False)

# Save minimal model outputs
output_timestamp = datetime.now().isoformat()
model_outputs = [
    {
        "question": q,
        "model_answer": p,
        "model_name": CONFIG["model_id"],
        "run_id": CONFIG["run_id"],
        "timestamp": output_timestamp
    }
    for q, p in zip(all_questions, predictions)
]

with open(output_dir / "model_outputs.json", "w") as f:
    json.dump(model_outputs, f, indent=2, ensure_ascii=False)

# Save confusion matrix
confusion_data = {
    "matrix": {f"true_{e}_pred_{p}": confusion.get((e, p), 0) 
               for e in [1, 2, 3] for p in [1, 2, 3]},
    "labels": [1, 2, 3]
}
with open(output_dir / "confusion.json", "w") as f:
    json.dump(confusion_data, f, indent=2)

print(f"\nArtifacts saved to: {output_dir}")
print("- metrics.json")
print("- config.json")
print("- notes.md")
print("- detailed_results.json")
print("- model_outputs.json")
print("- confusion.json")